In [ ]:
import os, sqlite3, warnings
import numpy as np, pandas as pd
import seaborn as sns, matplotlib.pyplot as plt
import plotly.express as px, plotly.graph_objects as go
from pathlib import Path
warnings.filterwarnings("ignore")
BASE_DIR = Path(os.getcwd()).resolve().parent
DB_PATH = BASE_DIR / "data" / "db" / "bluestock_mf.db"
CHARTS_DIR = BASE_DIR / "charts"
CHARTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"DB exists: {DB_PATH.exists()}")

In [ ]:
RF_ANNUAL = 0.065
RF_DAILY = RF_ANNUAL / 252

In [ ]:
def query(sql):
    conn = sqlite3.connect(DB_PATH); df = pd.read_sql(sql, conn); conn.close(); return df

nav = query("SELECT date_id, amfi_code, nav FROM fact_nav ORDER BY date_id")
nav["date"] = pd.to_datetime(nav["date_id"])
pivot = nav.pivot(index="date", columns="amfi_code", values="nav").ffill()
daily_returns = pivot.pct_change().dropna()
print(f"Daily returns: {daily_returns.shape}")

In [ ]:
cagr = {}
for code in daily_returns.columns:
    s = pivot[code].dropna()
    if len(s) < 2: continue
    years = (s.index[-1] - s.index[0]).days / 365.25
    if years > 0: cagr[code] = (s.iloc[-1] / s.iloc[0]) ** (1 / years) - 1
cagr_s = pd.Series(cagr, name="CAGR").sort_values(ascending=False)
print("Top 10 by CAGR:"); print(cagr_s.head(10))

In [ ]:
from scipy.stats import linregress
mean_ret = daily_returns.mean()
std_ret = daily_returns.std()
downside = daily_returns[daily_returns < 0].std()
sharpe = (mean_ret - RF_DAILY) / std_ret * np.sqrt(252)
sortino = (mean_ret - RF_DAILY) / downside * np.sqrt(252)
sr = pd.DataFrame({"Sharpe": sharpe, "Sortino": sortino}).sort_values("Sharpe", ascending=False)
print("Top 10 by Sharpe:"); print(sr.head(10))

In [ ]:
bench = query("SELECT date_id, index_name, close_value FROM fact_benchmark")
bench["date"] = pd.to_datetime(bench["date_id"])
b = bench[bench["index_name"] == "NIFTY50"].set_index("date")["close_value"]
bench_ret = b.pct_change().dropna()
ab = {}
for code in daily_returns.columns:
    fr = daily_returns[code].dropna()
    a, bv = fr.align(bench_ret, join="inner")
    if len(a) < 30: continue
    slope, intercept, r, _, _ = linregress(bv, a)
    ab[code] = {"Alpha": intercept * 252, "Beta": slope, "R2": r ** 2}
ab_df = pd.DataFrame(ab).T.sort_values("Alpha", ascending=False)
print("Top 5 by Alpha:"); print(ab_df.head())

In [ ]:
mdd = {}
for code in daily_returns.columns:
    cum = (1 + daily_returns[code]).cumprod()
    mdd[code] = (cum / cum.cummax() - 1).min()
mdd_s = pd.Series(mdd, name="MDD").sort_values()
print("Worst 5 MDD:"); print(mdd_s.head())

In [ ]:
score = pd.DataFrame(index=daily_returns.columns)
score["CAGR"] = cagr_s
score["Sharpe"] = sharpe
score["Alpha"] = ab_df["Alpha"]
score["Beta"] = ab_df["Beta"]
score["MDD"] = mdd_s
n = len(score)
for col in score.columns:
    asc = col in ["Beta", "MDD"]
    score[f"{col}_Score"] = (n - score[col].rank(ascending=asc) + 1) / n * 100
score["Total"] = (0.30 * score["CAGR_Score"] + 0.25 * score["Sharpe_Score"]
                  + 0.20 * score["Alpha_Score"] + 0.15 * score["Beta_Score"]
                  + 0.10 * score["MDD_Score"])
score = score.sort_values("Total", ascending=False)
funds = query("SELECT amfi_code, scheme_name, fund_house FROM dim_fund")
score = score.reset_index().merge(funds, left_on="index", right_on="amfi_code")
print("Top 10 Scorecard:")
print(score[["scheme_name", "fund_house", "Total"]].head(10))
score.to_csv(BASE_DIR / "fund_scorecard.csv", index=False)

In [ ]:
top5 = score["amfi_code"].head(5).tolist()
cum_f = (1 + daily_returns[top5]).cumprod()
cum_b = (1 + bench_ret).cumprod()
name_map = dict(zip(funds["amfi_code"], funds["scheme_name"]))
fig = go.Figure()
for c in top5:
    fig.add_trace(go.Scatter(x=cum_f.index, y=cum_f[c], mode="lines", name=name_map.get(c, str(c))[:30]))
fig.add_trace(go.Scatter(x=cum_b.index, y=cum_b, mode="lines", name="NIFTY50",
                         line=dict(color="black", width=3, dash="dash")))
fig.update_layout(title="Top 5 Scorecard Funds vs NIFTY50 (Rebased)", template="plotly_white")
fig.write_image(str(CHARTS_DIR / "benchmark_comparison.png"), width=1400, height=700, scale=2)
fig.show()